In [21]:
#Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import shap
import random
import os

import nltk
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from nlpaug.augmenter.word import BackTranslationAug

from gensim.models import KeyedVectors 

D:\Terminal\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\Terminal\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\video_transcripts_enhanced.csv")
df.head()

,id,text,durasi
0,1.mp4,di sebelah saya sudah ada baik bj40 yang akan ...,0.59
1,2.mp4,Civic yang sudah dimodifikasi full carbon jadi...,1.33
2,3.mp4,nama tempat itu gua musang dan kita mampir sal...,1.06
3,4.mp4,bisa membuka Khazanah Khazanah nih bahasanya j...,2.00
4,5.mp4,HP 3 juta yang banyak gaya kotaknya aja nantan...,2.18


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      200 non-null    object 
 1   text    200 non-null    object 
 2   durasi  200 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.8+ KB


In [9]:
# Cek duplikat berdasarkan kolom 'id'
duplikat_text = df.duplicated(subset=['text'])
print("Duplikat berdasarkan text:")
print(duplikat_text)
print("Jumlah duplikat text:", duplikat_text.sum())

Duplikat berdasarkan text:
0      False
1      False
2      False
3      False
4      False
       ...  
195     True
196     True
197    False
198    False
199    False
Length: 200, dtype: bool
Jumlah duplikat text: 10


In [13]:
# Tampilkan duplikat berdasarkan ID
duplikat_text_all = df[df.duplicated(subset=['text'], keep=False)]
print("\nData dengan ID duplikat:")
print(duplikat_text_all)


Data dengan ID duplikat:
          id                                               text  durasi
17    18.mp4  is I know you have to make time to reflect wha...    0.51
25    26.mp4  is I know you have to make time to reflect wha...    0.51
26    27.mp4  is I know you have to make time to reflect wha...    0.51
27    28.mp4  is I know you have to make time to reflect wha...    0.51
65    66.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.57
89    90.mp4  koplingnya kayak motor bebekan ini mobil kopli...    1.11
92    93.mp4  koplingnya kayak motor bebekan ini mobil kopli...    1.11
114  115.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    1.07
127  128.mp4  Audio tidak dapat dikenali - kemungkinan tidak...   10.31
146  147.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.44
194  195.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.35
195  196.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.31
196  197.mp4  Audio tidak dapat dikena

In [15]:
# Dictionary id -> teks baru
replace_dict = {
    "66.mp4": "ini warna cokelat, ini warna hitam, maaf ya kak, ini dijelaskan dulu, ini warna fuschia, ini wanra silver, ini warna gold, ini warna cokelat, ini warna hitam, ini baru banget dan ini namanya, aman, oh ini lebih gelap ya, nah bagus ya",
    "115.mp4": "kejuaraan dunia voli, alhamdulillah, ini sejarah kita dipercaya menjadi tuan rumah dari kejuaraan dunia u21 untuk wanita dan ini adalah langkah awal, bagaimana kita akan mempersiapkan timnas kita, baik putra dan putri, untuk lebih baik lagi di tingkat dunia, dan hari ini kita menyaksikan indonesia lawan vietnam, semoga timnas indonesia lolos dari group, dan semoga sampe final, setuju? amin, saya harap para masyarakat pecinta volly khususnya daerah jawa timur, dari tanggal 7 sampai 17, kalau bisa setiap timnas indonesia main kita saksikan dan ramaikan, setuju?",
    "128.mp4": "iya baik, kembali lagi dengan saya, tentunya sekarang kita akan membahasa, badminton championship 2025, yang baru saja selesai, dan indonesia lolos quarter final di babak beregu dan indonesia merebut gelar juara di kategori perorangan tunggal putra melali ahmad, dan meraih runner up di kategori ganda campuran, selamat pak, karena telah juara di kategori tunggal putra, 24 tahun indonesia tidak pernah merebut gelar juara dan baru kali ini, indonesia mendapatkan gelar juara untuk kategori tunggal putra, sebelum kesana nih saya mau tanya dan akan saya wawancarai, nah kenapa beregu hanya sampe runner up, ada banyak faktor ya kenapa itu bisa terjadi",
    "147.mp4": "halo selamatt siang saya periksa dulu ya, tadi kakak udah mulai sesak ya, iya dok, oke saya periksa dulu ya, ooo iya iya, ini kayaknya ibu asma, kalo asma kita harus uap biar pernapasan makin enak, ibu bersedia gak, bersedia dok, bang minta tolong di uap ya, boleh, siap dok, ibu kita uap dulu ya pake apas, salah, salah, salah, bu kita uap dulu ya, pake ini, jadi asma itu bisa ditangani di puskesmas lo, ayo kita dateng ke puskesmas, ayo"
}

# Update dataframe
for vid, teks_baru in replace_dict.items():
    df.loc[df['id'] == vid, 'text'] = teks_baru


In [17]:
# Tampilkan duplikat berdasarkan ID
duplikat_text_all = df[df.duplicated(subset=['text'], keep=False)]
print("\nData dengan ID duplikat:")
print(duplikat_text_all)


Data dengan ID duplikat:
          id                                               text  durasi
17    18.mp4  is I know you have to make time to reflect wha...    0.51
25    26.mp4  is I know you have to make time to reflect wha...    0.51
26    27.mp4  is I know you have to make time to reflect wha...    0.51
27    28.mp4  is I know you have to make time to reflect wha...    0.51
89    90.mp4  koplingnya kayak motor bebekan ini mobil kopli...    1.11
92    93.mp4  koplingnya kayak motor bebekan ini mobil kopli...    1.11
194  195.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.35
195  196.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.31
196  197.mp4  Audio tidak dapat dikenali - kemungkinan tidak...    0.35


In [22]:
#Case Folding & cleaning teks sederhana
def clean_text(text):
    text = str(text).lower()  # case folding
    text = re.sub(r"http\S+|www\S+", "", text)  # hapus URL
    text = re.sub(r"@\w+", "", text)  # hapus mention
    text = re.sub(r"#\w+", "", text)  # hapus hashtag
    text = re.sub(r"[^a-zA-Z\s]", " ", text)  # hapus angka & simbol
    text = re.sub(r"\s+", " ", text).strip()  # hapus spasi berlebih
    return text

df['text_clean'] = df['text'].apply(clean_text) 

In [25]:
#Normalisasi Text
def load_kamus(file_path):
    kamus={}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                slang = parts[0]
                normal = " ".join(parts[1:])
                kamus[slang] = normal
    return kamus

In [27]:
def normalisasi_teks(teks, kamus):
    words = teks.split()
    return " ".join([kamus.get(w, w) for w in words])

In [29]:
kamus = load_kamus(r"C:\Users\Aditya P J\Documents\Kuliah\Skripsi\Data\kbba.txt")
df['normalisasi_teks'] = df['text_clean'].apply(lambda x: normalisasi_teks(x, kamus))
df.head()

,id,text,durasi,text_clean,normalisasi_teks
0,1.mp4,di sebelah saya sudah ada baik bj40 yang akan ...,0.59,di sebelah saya sudah ada baik bj yang akan ki...,di sebelah saya sudah ada baik bj yang akan ki...
1,2.mp4,Civic yang sudah dimodifikasi full carbon jadi...,1.33,civic yang sudah dimodifikasi full carbon jadi...,civic yang sudah dimodifikasi full carbon jadi...
2,3.mp4,nama tempat itu gua musang dan kita mampir sal...,1.06,nama tempat itu gua musang dan kita mampir sal...,nama tempat itu saya musang dan kita mampir sa...
3,4.mp4,bisa membuka Khazanah Khazanah nih bahasanya j...,2.00,bisa membuka khazanah khazanah nih bahasanya j...,bisa membuka khazanah khazanah ini bahasanya j...
4,5.mp4,HP 3 juta yang banyak gaya kotaknya aja nantan...,2.18,hp juta yang banyak gaya kotaknya aja nantang ...,hp juta yang banyak gaya kotaknya saja nantang...


In [33]:
# Save ke CSV
df.to_csv("data_test.csv", index=False, encoding="utf-8")